# Retrieval Augmented Generation (RAG) with Transformers

## Project Overview

This project demonstrates a full **Retrieval Augmented Generation (RAG)** pipeline using open source tools in Python.

This project:
1. **Retrieves** relevant passages from a corpus using dense vector embeddings.
2. **Augments** a language model with those passages as context.
3. **Generates** answers using a decoder only transformer (GPT model).


Components:

- SQuAD v2 dataset to serve as the foundation for training and testing the retrieval and generation components.

- A **SentenceTransformer encoder** (all-MiniLM-L6-v2) to embed document chunks.

- **FAISS** as a vector index for fast semantic search over 19k unique passages.

- A **Mistral Model** to generate answers conditioned on retrieved context.

## 1) Installing and Importing Required Packages

In [18]:
!pip install -q sentence-transformers
!pip install faiss-cpu
!pip install transformers accelerate bitsandbytes

import torch
import faiss
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers import AutoModelForSequenceClassification
import torch.nn.functional as F



## 2) Loading Sentence Transformer model: sentence-transformers/all-MiniLM-L6-v2

For the embedding model, I selected all-MiniLM-L6-v2 because MiniLM is a lightweight, efficient encoder transformer designed specifically for high quality semantic embeddings.

It is extremely fast on Google Colab and takes advantage of GPU acceleration when available and is widely used in real world RAG pipelines, tutorials, and vector search workflows, making it a reliable baseline model.

In [19]:
# Setting model to GPU (CUDA) if available.
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# Loading the SentenceTransformer encoder model
encoder_model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Setting model to the appropriate device (GPU).
encoder = SentenceTransformer(encoder_model_name, device=device)


## 3) Loading the  Dataset

For this project, I use the **SQuAD v2** dataset (rajpurkar/squad_v2) from Hugging Face.

Each row contains:
- context: a paragraph from a Wikipedia article  
- question: a natural language question about that paragraph  
- answers: one or more answers


The dataset is utilized as a **document corpus** for the RAG system:

- The context field becomes my "document chunk".
- The question field is my user query.

In [20]:
# Loading the SQuAD v2 dataset from Hugging Face.
ds = load_dataset("rajpurkar/squad_v2")

#Inspecting the dataset
ds


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

## 4) Preparing Dataset for Use in Model

SQuAD is structured so that each training example includes a context field  that represents a paragraph from Wikipedia. These paragraphs are chunks of information that are small enough (100–300 words) for a retrieval system. The dataset has 130k question–context pairs, with many of thes paragraphs appearing repeatedly under different questions.

To prepare the data for the RAG system I:

1. Extracted all context fields from the dataset  
2. Deduplicated them using a Python set()
3. Ended up with 19k unique Wikipedia paragraphs  

Chunking is not needed for this particular dataset due to the paragraphs being short enough for embedding, topically consistent, and deigned to be used as QA passages.

In [21]:
# Extracting every context paragraph from the SQuAD dataset.SQuAD pairs each question with a context paragraph, but many questions reusethe same paragraph. This step gives us 130k rows with duplicate contexts.
all_contexts = [row["context"] for row in ds["train"]]
len(all_contexts)

# Deduplicating the paragraphs to get unique document chunks. In SQuAD, the same Wikipedia paragraph can appear 5–20+ times for different questions. For a RAG knowledge base, we only want one copy of each paragraph.
unique_contexts = list(set(all_contexts))
len(unique_contexts)


19029

## 5) Encoding Context Data with the Transformer Encoder

Next I converted each paragraph into a numerical embedding. Large Language Models and vector databases cannot operate directly
on raw text. They rely on embeddings, which are mathematical representations of meaning.

To create these embeddings, I use the **SentenceTransformer** model
all-MiniLM-L6-v2. This is an encoder only transformer that produces high quality 384 dimensional semantic vectors. These embeddings
capture the meaning of each paragraph well enough to compare questions to
documents based on similarity.


Calling encoder.encode(contexts) performs the following:

1. **Tokenization**  
   - Each paragraph is broken into tokens.  
   - Tokens are converted into integer IDs.

2. **Embedding Lookups**  
   - Token IDs are mapped to learned token embeddings.
   - Positional embeddings are added so the model knows word order.

3. **Transformer Encoder Forward Pass**  
   - The tokens pass through multiple layers of **unmasked self-attention**.
   - Each token attends to *every other token* in the paragraph.
   - This allows the model to build a deep semantic understanding of the text.

4. **Pooling to Create a Sentence Vector**  
   - All token embeddings are combined into a **single fixed-size vector**
     (MiniLM uses mean pooling by default).
   - Output is a 384-dimensional embedding.


These embeddings form the foundation of the retrieval system.


In [22]:
# Utilizing the deduplicated list of Wikipedia paragraphs as document corpus
contexts = unique_contexts

# Encoding all context paragraphs into semantic embeddings. Returning a numpy array (input needed for FAISS)
context_embeddings = encoder.encode(
    contexts,
    convert_to_numpy=True,
    show_progress_bar=True
)
context_embeddings.shape



Batches:   0%|          | 0/595 [00:00<?, ?it/s]

(19029, 384)

## 6) Normalizing Embeddings and Building the FAISS Vector Index

Next I normalized the embeddings before adding them to the FAISS index. Sentence transformer models like MiniLM are trained using **cosine similarity**, which measures how closely two vectors point in the same direction. However, FAISS does not provide a native cosine similarity index. FAISS only supports inner product or Euclidean distance. By normalizing each embedding so it has a length of 1, the **inner product becomes mathematically equivalent to cosine similarity**. This allows IndexFlatIP to effectively perform cosine based semantic search, ensuring that FAISS retrieves paragraphs that are genuinely closest in meaning to the user’s query.


**FAISS (Facebook AI Similarity Search)** is a high performance library built for fast vector similarity search. I added the normalized embeddings to FAISS creating a searchable vector index. The vector index acts as the knowledge memory of our RAG system.

In [23]:
#normalizing embeddings for input into vector index
normalized_embeddings = context_embeddings / np.linalg.norm(
    context_embeddings, axis=1, keepdims=True
)

# Checking the shape of the normalized embedding matrix.
normalized_embeddings.shape



(19029, 384)

In [24]:
# Determinining the dimensionality of the embeddings
dimension = normalized_embeddings.shape[1]  # 384

# Creating a FAISS index using Inner Product (IP) similarity.
index = faiss.IndexFlatIP(dimension)

# Adding all normalized context embeddings into the FAISS index.
index.add(normalized_embeddings)

# Printing the number of vectors stored in the index to confirm it loaded correctly.
print("Number of vectors in index:", index.ntotal)


Number of vectors in index: 19029


## 7) Retrieval Function: Query → Top-k Relevant Contexts

With the document corpus embedded and stored inside the FAISS vector index, I implemented the retrieval component of the RAG pipeline through the retrieve(query, k) function.

The retrieve(query, k) function takes a natural language question and returns
the **top-k most semantically relevant context paragraphs** from the knowledge
base. It works in four steps:


1. **Encode the query**  
   The user’s question is passed through the same SentenceTransformer encoder
   that was used for the corpus (all-MiniLM-L6-v2).

2. **Normalize the query embedding**  
   Just like the document embeddings, the query vector is normalized so that  
   **inner product = cosine similarity**.

3. **Search the FAISS vector index**  
   FAISS performs a fast nearest neighbor search over all 19,029 context vectors
   and returns the top-k most similar results.

4. **Return results with metadata**  
   The function returns:
   - the similarity score  
   - the index of the matching context  
   - the raw paragraph text  



These retrieved paragraphs form the **grounding evidence** the LLM will use
during the generation phase.

In [25]:
def retrieve(query, k=5):
    """
    Steps:
      1. Encode the query into the same embedding space as the documents.
      2. Normalize the query embedding so inner product ≈ cosine similarity.
      3. Perform vector search using FAISS to find nearest neighbors.
      4. Return the matching context paragraphs with similarity scores.
    """

    # Converting the text query into a 384-dimensional embedding vector.
    query_emb = encoder.encode([query], convert_to_numpy=True)

    # Normalizing the embedding so that inner product becomes equivalent to cosine similarity.
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)

    # Searching the FAISS index and returning the similiary score and indices of top-k most similar document embeddings
    scores, idxs = index.search(query_emb, k)

    # Returning results
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({
            "score": float(score),
            "context_index": int(idx),
            "context": contexts[idx],
        })

    return results


In [26]:
# Example query to test the retrieval system.
test_query = "When was the university founded?"

# Retrieving the top 3 most semantically similar context paragraphs.
results = retrieve(test_query, k=3)

# Printing the position, similarity score and first 400 characters of context paragraph
for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score={r['score']:.3f}) ---\n")
    print(r["context"][:400], "...")


--- Result 1 (score=0.551) ---

An important idea in the definition of a university is the notion of academic freedom. The first documentary evidence of this comes from early in the life of the first university. The University of Bologna adopted an academic charter, the Constitutio Habita, in 1158 or 1155, which guaranteed the right of a traveling scholar to unhindered passage in the interests of education. Today this is claimed ...

--- Result 2 (score=0.529) ---

Yale University is an American private Ivy League research university in New Haven, Connecticut. Founded in 1701 in Saybrook Colony as the Collegiate School, the University is the third-oldest institution of higher education in the United States. The school was renamed Yale College in 1718 in recognition of a gift from Elihu Yale, who was governor of the British East India Company. Established to  ...

--- Result 3 (score=0.528) ---

Greeks have a long tradition of valuing and investing in paideia (education). Paideia was o

## 8) Loading the Decoder LLM (Generator Model)

Next, to generate answers, I  used a decoder transformer, mistralai/Mistral-7B-Instruct-v0.3. Decoder only models are designed for autoregressive generation. They read all the input tokens (including context paragraphs + question), use **masked self attention** so each token can only attend to previous tokens, and predict the next token one at a time to build a coherent answer.


I utilized the Mistral model here because it fits on a Colab GPU while still offering strong reasoning and grounded generation.

This step completes the **RAG architecture**.

In [27]:
# Specifying the decoder model Mistral-7B-Instruct.
decoder_model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# Loading the tokenizer for Mistral for turning text into token IDs understood by the model.
tokenizer = AutoTokenizer.from_pretrained(decoder_model_name)

# Setting pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Loading the Mistral-7B-Instruct model and placing on GPU
decoder_model = AutoModelForCausalLM.from_pretrained(
    decoder_model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

# Creating a text generation pipeline that wraps tokenization, model forward pass, decoding of output tokens back into text
text_generator = pipeline(
    "text-generation",
    model=decoder_model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [28]:
# Before connecting to the full RAG pipeline, testing by itself to ensure text generation works correctly
out = text_generator(
    "The history of universities begins",
    max_new_tokens=30,
)

# Printing generated text to verify Mistral is working.
print(out[0]["generated_text"])


The history of universities begins in ancient Greece, with the establishment of the Platonic Academy in 387 BC. The first modern university was the University of Bolog


## 9) RAG Prompt Construction

Next, I built a custom RAG prompt. To make the LLM produce grounded, factual answers. I constructed a structured prompt that includes:

1. **Retrieved context paragraphs**  
   These come from the FAISS vector index and represent the most semantically
   relevant knowledge for the user’s query.

2. **Clear instructions**  
   The prompt tells the model to answer the question *using only the provided
   context*, which reduces hallucinations.

3. **The user’s question**  
   Placed after the context so the LLM can condition on both the retrieved
   evidence and the query.



In [29]:
def build_rag_prompt(query, retrieved_docs):
    """
    Construct a structured prompt for the Mistral-7B-Instruct decoder LLM that includes:
      - Retrieved context paragraphs (external knowledge)
      - Instructions for grounded answering
      - The user's question

    The prompt is wrapped in Mistral's [INST] ... [/INST] format so the model
    treats everything inside as the user instruction, and then generates the
    answer after [/INST].
    """

    # Starting with Mistral's instruction format.
    prompt = (
        "<s>[INST] "
        "You are a helpful AI assistant. Use ONLY the information in the context below to answer the question. Do not use prior knowledge. Do not guess. If the answer cannot be found in the context, say that you do not know.\n\n"
    )

    # Adding the retrieved context paragraphs.
    prompt += "### CONTEXT ###\n"
    for i, doc in enumerate(retrieved_docs, 1):
        prompt += f"Context {i}:\n{doc['context']}\n\n"

    # Adding the user's question after the context.
    prompt += "### QUESTION ###\n"
    prompt += query + "\n\n"

    # Closing the instruction block.
    prompt += "[/INST]\n"

    # Adding an ANSWER header to make the output structure clear.
    prompt += "### ANSWER ###\n"

    return prompt


In [30]:
#Testing the pipeline with an example question
query = "When was Yale University founded?"

# Retrieving the top 3 most relevant context paragraphs.
retrieved = retrieve(query, k=3)

# Building the RAG prompt using the retrieved evidence.
prompt = build_rag_prompt(query, retrieved)

# Printing the first 1200 characters of the prompt for inspection to verify context sections are included correctly, question is placed in the right spot and the prompt formatting is clean and readable
print(prompt[:1200])


<s>[INST] You are a helpful AI assistant. Use ONLY the information in the context below to answer the question. Do not use prior knowledge. Do not guess. If the answer cannot be found in the context, say that you do not know.

### CONTEXT ###
Context 1:
Yale University is an American private Ivy League research university in New Haven, Connecticut. Founded in 1701 in Saybrook Colony as the Collegiate School, the University is the third-oldest institution of higher education in the United States. The school was renamed Yale College in 1718 in recognition of a gift from Elihu Yale, who was governor of the British East India Company. Established to train Congregationalist ministers in theology and sacred languages, by 1777 the school's curriculum began to incorporate humanities and sciences. In the 19th century the school incorporated graduate and professional instruction, awarding the first Ph.D. in the United States in 1861 and organizing as a university in 1887.

Context 2:
Yale traces

## 10) Implementing a Hallucination Guardrail Using Natural Language Inference (NLI)

Large Language Models can produce answers that appear correct but are not actually supported by the retrieved context. To reduce hallucination and improve reliability, I implemented a Natural Language Inference (NLI) guardrail that verifies whether the model’s generated answer is logically supported by the retrieved evidence.

Natural Language Inference (NLI) is a task where a model evaluates the relationship between two pieces of text:

- **Premise** — a retrieved context passage  
- **Hypothesis** — the model’s generated answer  

The NLI model classifies the relationship as:

- **Entailment** → context supports the answer  
- **Contradiction** → context refutes the answer  
- **Neutral** → context provides no evidence

In [31]:

#Loading the NLI model
nli_model_name = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)

In [32]:
def nli_entailment_score(premise, hypothesis):
    """
    Use the NLI model to determine whether the hypothesis (answer)
    is logically supported by the premise (retrieved context).

    Returns:
        entail_prob (float): probability that the answer is entailed by the context.
    """
    # Tokenize the pair: premise = context, hypothesis = answer
    inputs = nli_tokenizer.encode_plus(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True
    ).to(device)

    # Run the NLI model without gradients
    with torch.no_grad():
        logits = nli_model(**inputs).logits

    # Convert logits to probabilities
    probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    # MNLI label order: [contradiction, neutral, entailment]
    entail_prob = probs[2]

    return float(entail_prob)


## 11) RAG Answer Function with NLI Hallucination Guardrail

In this final stage, I integrate the entire RAG pipeline into a single function.  

The full workflow consists of the following steps:

### **1. Retrieve the Most Relevant Context**
The user’s question is embedded using the MiniLM encoder.  
FAISS performs a vector similarity search and retrieves the top-k most relevant SQuAD passages.  
These passages serve as the external knowledge source for the LLM.

### **2. Construct a RAG Prompt with Mistral Instruction Format**
Using build_rag_prompt(), the system constructs a structured prompt that contains:

- The retrieved context paragraphs  
- An instruction telling the model to rely **only** on the retrieved evidence  
- The user’s question  
- A dedicated answer section  

The prompt is wrapped in Mistral’s [INST] format so the model interprets everything inside [INST] as the user instruction.

### **3. Generate an Answer Using Mistral-7B-Instruct**
The prompt is passed into the decoder model (Mistral-7B-Instruct).  
Using **masked self attention**, the LLM generates a grounded answer based on the retrieved context.

### **4. Verify the Answer Using an NLI Model (Hallucination Detection)**
After generation, the answer is checked using a Natural Language Inference model (bart-large-mnli). For each retrieved passage:

- The **context** acts as the *premise*  
- The **generated answer** acts as the *hypothesis*  

The NLI model predicts whether the hypothesis is:

- **Entailed** (supported)  
- **Contradicted** (refuted)  
- **Neutral** (not supported)  

If **no** retrieved context entails the generated answer above a chosen threshold (0.5), the system determines that the answer is a hallucination.

### **5. Apply the Guardrail**
If hallucination is detected:

- The system **does not return the model's answer**, and instead responds with:  
  *“I do not know. The retrieved context does not support the answer.”*

If the answer is supported:

- The original generated answer is returned.

---

### **6. Output the Full Pipeline Result**
The function returns:

- The final answer (guardrail applied)  
- The generated (raw) answer  
- The constructed RAG prompt  
- The retrieved contexts  
- The NLI entailment score  
- A boolean indicating whether the answer was supported

In [33]:
def rag_answer_with_nli_guardrail(query, k=3, max_new_tokens=100, entailment_threshold=0.50):
    """
    Full RAG pipeline + NLI hallucination guardrail.
    """
    retrieved = retrieve(query, k=k)
    prompt = build_rag_prompt(query, retrieved)

    outputs = text_generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    full_text = outputs[0]["generated_text"]
    raw_answer = full_text[len(prompt):].strip()

    # Evaluate groundedness via NLI
    max_entail = max(
        nli_entailment_score(doc["context"], raw_answer)
        for doc in retrieved
    )

    supported = max_entail >= entailment_threshold

    final_answer = raw_answer if supported else \
        "I do not know. The retrieved context does not support the answer."

    return {
        "query": query,
        "retrieved": retrieved,
        "prompt": prompt,
        "raw_answer": raw_answer,
        "max_entailment": max_entail,
        "supported": supported,
        "final_answer": final_answer,
    }



In [34]:
# Testing the Full Rag Pipeline

query = "When was Yale University founded?"
result = rag_answer_with_nli_guardrail(query)

print("Answer:", result["final_answer"])
print("Entailment score:", result["max_entailment"])
print("Supported:", result["supported"])


Answer: Yale University was founded in 1701.
Entailment score: 0.9909613728523254
Supported: True


## Conclusion

In this project, I built a complete **RAG** pipeline from scratch using open source tools. This system demonstrates how modern LLM applications can be grounded in external knowledge rather than relying solely on pretraining.


With this pipeline, a user can ask any question and the system will:

1. **Embed the query** using an encoder  transformer (MiniLM)
2. **Retrieve the most relevant context paragraphs** from a knowledge base using FAISS
3. **Construct a structured prompt** that combines:
   - Retrieved evidence  
   - Task instructions  
   - The user’s question  
4. **Generate a grounded answer** using a decoder transformer (Mistral)